# UHVDB r4 metadata + protein annotations

Build full r5-schema tables for UHVDB release 4 from the packaged figure_1 pipeline outputs.

**Primary inputs**
- Final cumulative release: `figure_1/uhvdb_human_metag_results/uhvdb_2026-03-26-2/`
- Source provenance: `figure_1/uhvdb_final_metadata.tsv`
- Lifestyle: `figure_3/uhvdb_v4_lifestyle.tsv`

**Outputs**
- `figure_3/uhvdb_r4_metadata.tsv.gz`
- `figure_3/uhvdb_r4_protein_annotations.tsv.gz`

Logic is adapted from `toolkit/bin/uhvdb_build_metadata.py`.


In [ ]:
import polars as pl

from build_uhvdb_r4_metadata import (
    OUT_PATH,
    RELEASE,
    SOURCE_META,
    add_host_predictions,
    add_lifestyle_and_integration,
    add_protein_aggregates,
    add_taxonomy,
    compare_to_r5,
    create_base_metadata,
    finalize_and_write,
)

print("RELEASE:", RELEASE)
print("SOURCE_META:", SOURCE_META)
print("OUT_PATH:", OUT_PATH)


## 1. Base metadata (IDs, clustering, classify, HQ/HC filters)

Join source provenance → seqhash → UHVDB mapping → genomovar/species/AAI clusters → classify/hqfilter/hcfilter.


In [ ]:
base = create_base_metadata()
base.head()


## 2. ICTV taxonomy + CRISPR/PHIST hosts

Taxonomy and host predictions are attached at the genomovar-rep level. Host lineages use GTDB R226 via taxopy.


In [ ]:
with_tax = add_taxonomy(base)
with_host = add_host_predictions(with_tax)
with_host.select([
    "uhvdb_id", "genomovar_rep", "genomad_taxonomy", "ictv_family",
    "final_host_pred", "host_lineage",
]).head()


## 3. Lifestyle + integration_status

Lifestyle scores come from `uhvdb_v4_lifestyle.tsv`. `integration_status` is per-row: classify provirus/topology **or** the row's own `seq_name` in the figure_3d integrated-input set (not propagated from other genomovar members).


In [ ]:
with_life = add_lifestyle_and_integration(with_host)
with_life.select([
    "uhvdb_id", "genomovar_rep", "topology", "provirus",
    "virulent", "temperate", "phrog_integrases", "integration_status",
]).head()


## 4. Protein hallmark aggregates

Aggregate bakta / foldseek / InterProScan / CARD / VFDB / pharokka / phold / Empathi annotations per genomovar representative (from the final r4 release tables).


In [ ]:
with_prot = add_protein_aggregates(with_life)
with_prot.select([
    "uhvdb_id", "genomovar_rep", "num_proteins", "num_uniprot_ips",
    "num_tail", "num_capsid", "num_lysis", "mcp_hallmark", "terl_hallmark", "portal_hallmark",
]).filter(pl.col("uhvdb_id") == pl.col("genomovar_rep")).head()


## 5. Write r4 metadata + basic sanity checks


In [ ]:
r4 = finalize_and_write(with_prot)
print("Output exists:", OUT_PATH.exists(), "size_mb=", round(OUT_PATH.stat().st_size / 1e6, 1))


## 6. Cross-check r4 genomovar_rep annotations vs r5

For canonical genomovar reps (`uhvdb_id == genomovar_rep` and `seq_name == seqhash_rep`) that also exist in r5, annotation columns should match at ~100%. Clustering IDs may differ after the r5 increment and are reported separately.


In [ ]:
compare_to_r5(r4)


## 7. Build r4 protein annotations

Per-protein table matching the r5 schema. Annotation columns are rebuilt from the r4 release tables; `start`/`end`/`strand`/`partial` are recovered from r5 for overlapping `protein_id`s (gene coordinates are not in the packaged release TSVs).


In [ ]:
from build_uhvdb_r4_protein_annotations import (
    OUT_PATH as PROTEIN_OUT_PATH,
    build_protein_annotations,
    add_coords_from_r5,
    compare_to_r5 as compare_proteins_to_r5,
    get_genomovar_reps,
    write_annotations,
)

reps = get_genomovar_reps()
protein_annots = build_protein_annotations(reps)
protein_annots = add_coords_from_r5(protein_annots)
protein_annots.head()


In [ ]:
write_annotations(protein_annots)
print("Output:", PROTEIN_OUT_PATH, "size_mb=", round(PROTEIN_OUT_PATH.stat().st_size / 1e6, 1))


## 8. Cross-check r4 protein annotations vs r5

For overlapping `protein_id`s, annotation and coordinate columns should match r5.


In [ ]:
compare_proteins_to_r5(protein_annots)
